# Round42 Seed Sweep

Renderiza o mesmo prompt de cada target em muitas seeds diferentes.

Isto testa variacao estocastica do modelo sem mudar o texto.

In [5]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "search_seed_sweep.py").exists():
    PROJECT_ROOT = Path("C:\\Users\\tugap\\Desktop\\Universidade\\Masters2\u00baAno\\IAG\\ProjetoCunha\\Projeto2")

SRC_DIR = PROJECT_ROOT / "src"
PROMPT_BANK = PROJECT_ROOT / "prompts" / "refinement_round42_seed_sweep_best.json"
TARGETS_DIR = PROJECT_ROOT / "TP2-students" / "students" / "tp2-chosen"
OUTPUT_DIR = PROJECT_ROOT / "TP2-students" / "students" / "outputs"

PYTHON_CANDIDATES = [
    PROJECT_ROOT / ".venv_win" / "Scripts" / "python.exe",
    Path("C:\\Users\\tugap\\Desktop\\Universidade\\Masters2\u00baAno\\IAG\\Projeto 2\\.venv\\Scripts\\python.exe"),
    Path(sys.executable),
]

def has_module(python_exe, module_name):
    if not Path(python_exe).exists():
        return False
    result = subprocess.run(
        [str(python_exe), "-c", f"import {module_name}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    return result.returncode == 0

PYTHON_EXE = None
for candidate in PYTHON_CANDIDATES:
    if has_module(candidate, "diffusers"):
        PYTHON_EXE = candidate
        break

if PYTHON_EXE is None:
    raise RuntimeError("No Python with diffusers found. Create .venv_win and install requirements.txt")

print("Project root:", PROJECT_ROOT)
print("Render/search Python:", PYTHON_EXE)
assert (SRC_DIR / "generate_round42_seed_sweep_prompts.py").exists()
assert (SRC_DIR / "search_seed_sweep.py").exists()


Project root: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2
Render/search Python: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe


## Configuracao

In [6]:
RUN_MODE = "metric_64"  # smoke | metric_64 | all_64 | metric_128

MODES = {
    "smoke": {
        "identity": "round42_seed_sweep_smoke",
        "prompt_source": "metric_best",
        "seed_offsets": list(range(0, 4)),
        "top_k": 4,
    },
    "metric_64": {
        "identity": "round42_seed_sweep_metric64",
        "prompt_source": "metric_best",
        "seed_offsets": list(range(0, 64)),
        "top_k": 12,
    },
    "all_64": {
        "identity": "round42_seed_sweep_all64",
        "prompt_source": "all",
        "seed_offsets": list(range(0, 64)),
        "top_k": 12,
    },
    "metric_128": {
        "identity": "round42_seed_sweep_metric128",
        "prompt_source": "metric_best",
        "seed_offsets": list(range(0, 128)),
        "top_k": 16,
    },
}

config = MODES[RUN_MODE]
print("Selected mode:", RUN_MODE)
print("renders:", len(config["seed_offsets"]), "seeds per selected prompt")
print(json.dumps({k: v for k, v in config.items() if k != "seed_offsets"}, indent=2))

Selected mode: metric_64
renders: 64 seeds per selected prompt
{
  "identity": "round42_seed_sweep_metric64",
  "prompt_source": "metric_best",
  "top_k": 12
}


## Gerar banco de prompts

In [7]:
cmd = [str(PYTHON_EXE), str(SRC_DIR / "generate_round42_seed_sweep_prompts.py"), "--output", str(PROMPT_BANK)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
data = json.loads(PROMPT_BANK.read_text(encoding="utf-8"))
print({target: len(entries) for target, entries in data.items()})

Running: C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\generate_round42_seed_sweep_prompts.py --output c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round42_seed_sweep_best.json
{'1159_25.png': 2, '1159_29.png': 2, '1159_3.png': 2, '1159_7.png': 1, '7836.png': 2, '9338.png': 1}


## Correr seed sweep

In [8]:
args = [
    str(PYTHON_EXE),
    str(SRC_DIR / "search_seed_sweep.py"),
    "--prompts", str(PROMPT_BANK),
    "--targets", str(TARGETS_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--identity", config["identity"],
    "--prompt-source", config["prompt_source"],
    "--top-k", str(config["top_k"]),
    "--seed-offsets", *[str(seed) for seed in config["seed_offsets"]],
    "--offline",
    "--disable-progress-bar",
    "--maxstack-scoring",
]

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

print("Running:")
print(" ".join(args[:12]), "...", len(config["seed_offsets"]), "seed offsets")
process = subprocess.Popen(
    args,
    cwd=PROJECT_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Seed sweep failed with exit code {return_code}")
print("Finished successfully")

Running:
C:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\Projeto 2\.venv\Scripts\python.exe c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\src\search_seed_sweep.py --prompts c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\prompts\refinement_round42_seed_sweep_best.json --targets c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\tp2-chosen --output-dir c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs --identity round42_seed_sweep_metric64 --prompt-source metric_best ... 64 seed offsets
Couldn't connect to the Hub: Cannot reach https://huggingface.co/api/models/SimianLuo/LCM_Dreamshaper_v7: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable..
Will try to load from local cache.

Loading pipeline components...: 100%|██████████| 7/7 [00:22<00:00,  3.18s/it]
Setting up [LPIPS] perceptual 

## Ver resultados

In [9]:
run_dirs = sorted(
    [path for path in OUTPUT_DIR.glob(f"*_{config['identity']}") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not run_dirs:
    print("No run directory found")
else:
    latest = run_dirs[0]
    print("Latest run:", latest)
    for item in sorted(latest.glob("*")):
        print(item.name)

Latest run: c:\Users\tugap\Desktop\Universidade\Masters2ºAno\IAG\ProjetoCunha\Projeto2\TP2-students\students\outputs\20260603-182138_round42_seed_sweep_metric64
1159_25
1159_25_seed_sweep_metrics.csv
1159_29
1159_29_seed_sweep_metrics.csv
1159_3
1159_3_seed_sweep_metrics.csv
1159_7
1159_7_seed_sweep_metrics.csv
7836
7836_seed_sweep_metrics.csv
9338
9338_seed_sweep_metrics.csv
contact_sheet_top12_seed_sweep.jpg
seed_sweep_metrics.csv
summary.json
top12_seed_sweep.csv
